# 11 深度學習（PyTorch）：用神經網路預測感染

同事問：「要不要試試深度學習？」
我們用 PyTorch 建一個簡單的二元分類網路，看看 280 筆資料上能做到什麼程度。

流程：**資料前處理 → 模型架構 → 訓練迴圈 → 早停法 → AUC 評估 → 學習曲線 → 與 sklearn 比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — 資料前處理：把病歷表格轉成 PyTorch 看得懂的張量

PyTorch 不吃 DataFrame，只吃 **tensor**（張量）。這步要把類別欄位轉成數字、把數值欄位標準化、確定型別是 `float32`，最後手動切出訓練集和驗證集——這些在 sklearn 裡通常是一行 `Pipeline` 搞定的事，這裡刻意拆開來手動做一遍，看清楚每一步在幹嘛。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)` | 建立二元結果變項（0/1）；跟 Ch10 一樣不用症狀當特徵，避免 data leakage |
> | `X_df = pd.get_dummies(df[...], drop_first=True)` | 類別欄位（`sex`、`wing`...）one-hot 編碼成 0/1 虛擬變數，`drop_first=True` 避免共線性 |
> | `X_np = X_df.values.astype(np.float32)` | 轉成 NumPy 陣列並轉型為 **`float32`**——PyTorch 的權重預設是單精度浮點，型別要對齊，不然會報 dtype 錯誤 |
> | `scaler.fit_transform(X_np[:, 0:1])` | 只對 `age` 做標準化（均值 0、標準差 1）；神經網路對輸入尺度敏感，不標準化會拖慢甚至搞砸收斂 |
> | `np.random.shuffle(idx)` → `train_idx, val_idx = idx[:split], idx[split:]` | 手動打亂索引、切 70/30 訓練 / 驗證集——這裡不用 `sklearn.train_test_split`，是刻意示範切分的原理 |
> | `y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)` | `unsqueeze(1)` 把形狀從 `(N,)` 變成 `(N, 1)`，對齊模型輸出的形狀，否則 loss function 會用 broadcasting 硬湊、算出錯誤的 loss |

> ⚠️ **小資料的隱藏 leakage**：這裡先在**全部 280 筆**上 `fit` 了 `StandardScaler`，才切訓練 / 驗證集——嚴格說，驗證集的統計量已經悄悄「滲入」了標準化參數，是一種很輕微的 data leakage。280 筆、只有一個數值欄位，滲漏小到可忽略，這裡純粹示範手動流程；真正的專案請學 Ch10，把前處理放進 `Pipeline`，只在訓練折上 `fit`。也因為切分是隨機的，開頭的 `torch.manual_seed(42)` / `np.random.seed(42)` 讓每次重跑都拿到一樣的切分和初始權重，結果才能重現、才能公平地跟 sklearn 比較。

In [ ]:
# --- Step 1: 資料前處理（手動轉 tensor）---
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 與 Ch10 相同的特徵（不用症狀，避免 data leakage）
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

# One-hot 編碼類別特徵
X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
# 轉成 NumPy 陣列並轉型為 float32（PyTorch 權重預設是單精度浮點，型別要對齊）
X_np = X_df.values.astype(np.float32)
y_np = df["infected"].values.astype(np.float32)

# 標準化 age
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

# 70/30 split
idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

# unsqueeze(1) 把 y 從一維 (N,) 轉成二維 (N, 1)，對齊模型輸出的形狀
X_train = torch.tensor(X_np[train_idx])
y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
X_val = torch.tensor(X_np[val_idx])
y_val = torch.tensor(y_np[val_idx]).unsqueeze(1)

print(f"特徵維度：{X_train.shape[1]}")
print(f"訓練集：{len(X_train)}，驗證集：{len(X_val)}")
print(f"特徵名稱：{list(X_df.columns)}")

## Step 2 — 模型架構：三層線性層疊出一個最小的神經網路

`nn.Sequential` 是 PyTorch 最簡單的容器，把層一個接一個串起來，資料由上往下依序通過。這裡疊了兩組「線性轉換 + ReLU 非線性」，最後一層直接輸出一個數字（logit），不接 sigmoid。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))` | 把層堆疊成一條管線：`Linear` 做線性組合，`ReLU` 加入非線性，最後 `Linear(16, 1)` 輸出單一數值 |
> | `nn.ReLU()` | 非線性 activation——少了它，多層 `Linear` 疊再多層，數學上等價於一層線性迴歸 |
> | `n_params = sum(p.numel() for p in model.parameters())` | 加總所有權重 + 偏差的元素數，量化這個模型「有多少東西可以調」 |

> 💡 **為什麼最後一層不接 sigmoid？** 因為 Step 3 會用 `nn.BCEWithLogitsLoss`，它把 sigmoid 和 binary cross-entropy 合併在內部計算，數值上比「先手動 sigmoid 再算 BCE」更穩定（避免 `log(0)` 這種極端值炸開）。所以模型只管輸出 raw logit，sigmoid 留到算機率（Step 5）或算 loss（Step 3）時才登場。

In [ ]:
# --- Step 2: 模型架構 ---
# input_dim \u2192 32 \u2192 16 \u2192 1
input_dim = X_train.shape[1]

# 只疊 Linear + ReLU，最後一層不接 activation（輸出 raw logit，交給 BCEWithLogitsLoss 處理）
model = nn.Sequential(
    nn.Linear(input_dim, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# 參數數量
n_params = sum(p.numel() for p in model.parameters())
print(f"模型架構：{input_dim} \u2192 32 \u2192 16 \u2192 1")
print(f"總參數量：{n_params}")
print(f"參數 / 樣本比：{n_params / len(X_train):.1f}")
print(f"\n\u2192 參數比樣本還多 \u2192 過擬合風險極高！")

## Step 3 — 訓練迴圈 + 早停法：3 個動詞跑最多 300 遍，但見好就收

PyTorch 的訓練迴圈永遠是同一個節奏：**forward**（算預測）→ **loss**（算誤差）→ **backward**（算梯度）→ **step**（更新參數）。這裡再加上早停法（early stopping）：一邊訓練一邊監控驗證集的 loss，只要連續 `patience` 輪都沒進步，就提早喊停、回頭用表現最好的那組權重。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `optimizer.zero_grad()` | 3 個核心動詞之一：清空上一輪殘留的梯度（PyTorch 預設梯度會累加，不清空會越加越大） |
> | `logits = model(X_train)` | forward：資料流過網路，得到 raw logits（還沒過 sigmoid） |
> | `loss = loss_fn(logits, y_train)` | 用 `BCEWithLogitsLoss` 比較預測和真實標籤，算出這一輪的誤差 |
> | `loss.backward()` | 3 個核心動詞之二：backprop，PyTorch 自動微分算出每個參數對 loss 的梯度 |
> | `optimizer.step()` | 3 個核心動詞之三：依剛算好的梯度更新每個參數（Adam optimizer 的規則） |
> | `if val_loss < best_val_loss: ... counter = 0 else: counter += 1` | 早停的核心：val_loss 進步就存檔、`counter` 歸零；沒進步就累加 `counter` |
> | `model.load_state_dict(best_state)` | 訓練結束後「倒帶」回 val_loss 最低的那次權重快照，而不是用最後一輪（可能已經 overfit）的權重 |

> 🧭 **見好就收**：`patience` 是「再給幾輪機會」，`best_state` 是「目前最佳成績的存檔點」。一旦連續 `patience` 輪都沒有刷新紀錄，就停下來把獎盃頒給最佳表現的那一輪，而不是頒給訓練結束當下（很可能已經開始 overfit）的自己。

In [ ]:
# --- Step 3: 訓練迴圈 + 早停法 ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 記錄歷史
train_losses, val_losses = [], []
best_val_loss = float("inf")
patience, counter = 15, 0
best_state = None
best_epoch = 0

for epoch in range(300):
    # 訓練
    model.train()
    # 3 個核心動詞之一：清空上一輪殘留的梯度
    optimizer.zero_grad()
    # forward：資料流過網路，得到 raw logits
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    # 3 個核心動詞之二：backprop，算出每個參數的梯度
    loss.backward()
    # 3 個核心動詞之三：依梯度更新參數
    optimizer.step()
    train_losses.append(loss.item())

    # 驗證
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
    val_losses.append(val_loss)

    # 早停
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # 存下目前最佳權重快照（clone 避免之後被覆寫）
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# 載入最佳模型
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")

## Step 4 — 學習曲線：把訓練過程畫出來

`train_losses` 和 `val_losses` 已經在 Step 3 迴圈裡逐輪記錄下來了，這裡只是把它們畫成兩條線，一眼看出訓練集和驗證集的落差如何隨 epoch 變化。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `ax.plot(train_losses, ...)` / `ax.plot(val_losses, ...)` | 把每個 epoch 記錄的 loss 畫成兩條線，比較訓練 / 驗證的走勢 |
> | `ax.axvline(x=best_epoch, ...)` | 標出早停法選中的最佳 epoch，方便對照曲線在那個時間點發生了什麼 |

> 💡 **看曲線抓 overfitting**：如果 train loss 一路下滑，但 val loss 先降後升、出現「V 型反彈」，那個反彈點就是模型開始死背訓練資料的訊號——早停法做的事，正是在反彈點附近喊停。

In [ ]:
# --- Step 4: 學習曲線 ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#2c7fb8")
ax.plot(val_losses, label="Val Loss", color="#e34a33")
# 標出早停法選中的最佳 epoch
ax.axvline(x=best_epoch, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Learning Curve")
ax.legend()
plt.tight_layout()
plt.show()

print("\u2192 如果 train loss 持續下降但 val loss 反彈 \u2192 過擬合")
print("\u2192 早停法在 val loss 不再改善時停止訓練")

## Step 5 — 評估：用驗證集算 AUC

訓練時餵給模型的是 raw logits，但 AUC 需要「像機率的分數」，所以評估時要手動補一次 `sigmoid`，把 logits 轉回 0~1 之間的機率，再用跟 Ch10 一樣的 `roc_auc_score` 打分數。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `with torch.no_grad():` | 評估階段不需要算梯度，關掉梯度追蹤可以省記憶體、加快計算 |
> | `torch.sigmoid(model(X_val)).numpy()` | 把模型輸出的 logits 過 sigmoid 轉成機率，才能丟進 `roc_auc_score` |
> | `roc_auc_score(y_val.numpy(), val_proba)` | 比較「機率排序」跟「真實標籤」，算出 AUC——跟 Ch10 用同一個指標，才能公平比較 |
> | `auc_train - auc_val` | 訓練集和驗證集 AUC 的差距，是最直接的 overfitting 偵測器 |

> ⚠️ **訓練用 logits，評估用機率**：`BCEWithLogitsLoss` 內部已經做了 sigmoid，訓練時直接餵 logits 給它效率更高、數值更穩定；但算 AUC 需要看得懂的機率分數，所以這裡才要自己補上 `torch.sigmoid()`。

In [ ]:
# --- Step 5: AUC 評估 ---
model.eval()
# 評估階段不需要算梯度，關掉梯度追蹤省記憶體、加快計算
with torch.no_grad():
    # 把 logits 過 sigmoid 轉成 0~1 的機率，才能丟進 roc_auc_score
    val_proba = torch.sigmoid(model(X_val)).numpy()
    train_proba = torch.sigmoid(model(X_train)).numpy()

auc_train = roc_auc_score(y_train.numpy(), train_proba)
auc_val = roc_auc_score(y_val.numpy(), val_proba)

print(f"=== PyTorch DL 結果 ===")
print(f"Train AUC = {auc_train:.3f}")
print(f"Val   AUC = {auc_val:.3f}")
print(f"Gap       = {auc_train - auc_val:.3f}")

if auc_train - auc_val > 0.1:
    print("\n\u2192 Train-Val gap > 0.1 \u2192 過擬合嚴重")
    print("\u2192 280 筆資料不足以支撐這個模型的參數量")
else:
    print("\n\u2192 Gap 不大，模型相對穩定")

## Step 6 — 與 sklearn 比較：DL 真的比較強嗎？

光看 DL 自己的 AUC 沒有意義，要跟 Ch10 的 Logistic Regression、Random Forest 放在**完全相同**的 train/val 切分下比較，才知道多堆幾層神經網路到底值不值得。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `X_full = df[num_cols + cat_cols + bin_cols]` | 用原始（未 one-hot、未標準化）欄位餵給 sklearn，前處理交給 `ColumnTransformer` 統一處理 |
> | `X_sk_train = X_full.iloc[train_idx]` | 沿用 Step 1 切好的**同一組** `train_idx` / `val_idx`，確保三個模型比較的是同一份考卷 |
> | `Pipeline([("pre", preprocess), ("model", LogisticRegression(...))])` | 跟 Ch10 一樣，把前處理和模型包成一條 Pipeline |
> | `compare.append(("PyTorch DL", auc_val))` | 把 Step 5 算好的 DL 驗證集 AUC 併入同一張比較表 |

> 🧭 **公平比較的關鍵是「相同切分」**：如果三個模型各自用不同的 train/val 切分，AUC 差異可能只是運氣（哪些人剛好分到驗證集），而不是模型本身的差別。這也是為什麼這裡執著地重複使用 `train_idx` / `val_idx`。

In [ ]:
# --- Step 6: 與 sklearn 比較 ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 用相同的 train/val split
X_full = df[num_cols + cat_cols + bin_cols]
y_full = df["infected"]

# 沿用 Step 1 同一組 train_idx / val_idx，確保三個模型比較的是同一份考卷
X_sk_train = X_full.iloc[train_idx]
X_sk_val = X_full.iloc[val_idx]
y_sk_train = y_full.iloc[train_idx]
y_sk_val = y_full.iloc[val_idx]

# 跟 Ch10 一樣：數值標準化 + 類別 one-hot + 二元直接通過
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

compare = []

# Logistic Regression
clf_lr = Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=500, random_state=42))])
clf_lr.fit(X_sk_train, y_sk_train)
auc_lr = roc_auc_score(y_sk_val, clf_lr.predict_proba(X_sk_val)[:, 1])
compare.append(("Logistic Regression", auc_lr))

# Random Forest
clf_rf = Pipeline([("pre", preprocess), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
clf_rf.fit(X_sk_train, y_sk_train)
auc_rf = roc_auc_score(y_sk_val, clf_rf.predict_proba(X_sk_val)[:, 1])
compare.append(("Random Forest", auc_rf))

# PyTorch
compare.append(("PyTorch DL", auc_val))

print("=== 模型比較（相同 train/val split）===")
for name, auc in compare:
    print(f"  {name:25s}  Val AUC = {auc:.3f}")

print("\n\u2192 在 280 筆資料上，三個模型的表現通常很接近")
print("\u2192 DL 並未展現明顯優勢，反而有過擬合風險")
print("\u2192 教學價值：學會 PyTorch 語法，未來遇到大資料集才能派上用場")

## 加碼：模型解釋——這個神經網路靠哪些線索做判斷？

DL 模型是個黑盒子，沒有像邏輯斯迴歸那樣的係數可以直接讀。這裡用**手動版 permutation importance**：把驗證集裡某一欄的值打亂洗牌，重新算一次 AUC，掉得越多，代表這一欄對模型越重要。跟 Ch10 用 `sklearn.inspection.permutation_importance` 是同一個概念，這裡自己動手實作一遍（也刻意不用 SHAP-on-torch——那個在無頭的 CI 環境跑起來慢又容易出錯，手動打亂法幾秒鐘就能跑完，穩定又不用額外依賴）。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `baseline_auc = roc_auc_score(y_val.numpy(), baseline_proba)` | 先算出「什麼都不打亂」的原始 AUC，當作比較基準 |
> | `for col_idx, col_name in enumerate(X_df.columns):` | 逐一欄位輪流打亂，每次只動一欄，其他欄位保持不變 |
> | `rng.shuffle(X_shuffled[:, col_idx])` | 把這一欄的值在列與列之間打亂，等於讓模型看到「這一欄的資訊被抹掉、其他欄位不變」的資料 |
> | `shuffled_auc = roc_auc_score(y_val.numpy(), shuffled_proba)` | 用打亂後的資料重新算 AUC |
> | `drops.append(baseline_auc - shuffled_auc)` | AUC 掉得越多，代表這一欄對模型的預測越重要；重複 `n_repeats` 次取平均，避免單次洗牌的運氣影響結果 |

> 🧭 **重要 ≠ 因果**：permutation importance 告訴你「模型有多依賴這條線索來預測」，不代表「改變這個特徵就能改變感染風險」。真正要問因果（例如淋浴暴露是否「導致」感染），要看 Ch06 校正干擾後的 adjusted OR，或 Ch12 的因果推論方法——預測力強的特徵，可能只是跟真正的病因一起出現的旁觀者。

In [ ]:
# --- 加碼: 手動計算 Permutation Importance ---
# 想法：打亂某一欄的值，如果這欄本來很重要，模型會「失憶」，AUC 明顯下降；
# 如果這欄根本沒用，打亂前後 AUC 幾乎不變。

rng = np.random.default_rng(42)
n_repeats = 10

model.eval()
with torch.no_grad():
    baseline_proba = torch.sigmoid(model(X_val)).numpy()
baseline_auc = roc_auc_score(y_val.numpy(), baseline_proba)

X_val_np = X_val.numpy()  # 轉回 NumPy，方便逐欄打亂
records = []

for col_idx, col_name in enumerate(X_df.columns):
    drops = []
    for _ in range(n_repeats):
        X_shuffled = X_val_np.copy()
        rng.shuffle(X_shuffled[:, col_idx])  # 只打亂這一欄，其他欄位不動
        with torch.no_grad():
            shuffled_proba = torch.sigmoid(model(torch.tensor(X_shuffled))).numpy()
        shuffled_auc = roc_auc_score(y_val.numpy(), shuffled_proba)
        drops.append(baseline_auc - shuffled_auc)
    records.append((col_name, float(np.mean(drops)), float(np.std(drops))))

imp_df = pd.DataFrame(records, columns=["feature", "importance", "std"])
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

print(f"Baseline Val AUC = {baseline_auc:.3f}\n")
print("=== Permutation Importance（AUC 下降量，越大越重要）===")
print(imp_df.head(8).to_string(index=False))

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 前處理 | `pd.get_dummies()` + `torch.tensor()` 手動轉換 |
| 模型 | `nn.Sequential(Linear \u2192 ReLU \u2192 Linear \u2192 ReLU \u2192 Linear)` |
| 訓練迴圈 | `zero_grad \u2192 forward \u2192 loss \u2192 backward \u2192 step` |
| 早停法 | 監控 val_loss，patience 到了就停 |
| 學習曲線 | 視覺化 train/val loss 診斷過擬合 |
| 模型比較 | 相同 split 下公平比較 DL vs sklearn |

**結論**：
- 280 筆 \u2192 DL 過殺，sklearn 就夠了
- 但 PyTorch 語法值得學：未來遇到影像、序列、大樣本就需要
- 重點不是「哪個模型最強」，而是「用正確的工具解決正確的問題」

下一章（Ch12），我們問：淋浴真的「導致」感染嗎？ \u2192 因果推論。